# Calibration, Uncertainty, and Distribution Shift Lab


## Purpose

This lab computes confidence-bin calibration and expected calibration error.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

n = 4000

evaluation = pd.DataFrame({
    "example_id": [f"ex_{i:05d}" for i in range(1, n + 1)],
    "split": rng.choice(
        ["train", "validation", "test", "shifted_deployment"],
        size=n,
        p=[0.45, 0.20, 0.20, 0.15],
    ),
})

base_difficulty = rng.beta(2.0, 5.0, size=n)

shift_penalty = np.where(
    evaluation["split"] == "shifted_deployment",
    0.18,
    0.0,
)

evaluation["true_label"] = rng.binomial(1, 0.45, size=n)

signal_strength = np.where(
    evaluation["true_label"] == 1,
    0.70,
    0.30,
)

noise = rng.normal(0, 0.14 + shift_penalty, size=n)

evaluation["predicted_probability"] = np.clip(
    signal_strength - 0.20 * base_difficulty + noise,
    0.01,
    0.99,
)

evaluation["prediction"] = (evaluation["predicted_probability"] >= 0.50).astype(int)
evaluation["correct"] = (evaluation["prediction"] == evaluation["true_label"]).astype(int)

evaluation.head()

In [ ]:
evaluation["confidence_bin"] = pd.cut(
    evaluation["predicted_probability"],
    bins=np.linspace(0, 1, 11),
    include_lowest=True,
)

calibration = (
    evaluation
    .groupby(["split", "confidence_bin"], observed=False)
    .agg(
        n=("example_id", "count"),
        accuracy=("correct", "mean"),
        confidence=("predicted_probability", "mean"),
    )
    .reset_index()
)

calibration["abs_calibration_gap"] = (
    calibration["accuracy"] - calibration["confidence"]
).abs()

ece_by_split = (
    calibration
    .dropna()
    .assign(weight=lambda df: df["n"] / df.groupby("split")["n"].transform("sum"))
    .assign(weighted_gap=lambda df: df["weight"] * df["abs_calibration_gap"])
    .groupby("split")
    .agg(expected_calibration_error=("weighted_gap", "sum"))
    .reset_index()
)

ece_by_split

## Interpretation

Calibration evaluates whether model confidence matches empirical correctness, especially under shifted deployment.